# Respiratory ICU mortality (eICU-CRD v2.0) - 10 sample records through the 8 bias steps

This notebook follows **10 real training records** (chosen at random, fixed seed) through every preprocessing step of
`respiratory_preprocessing.ipynb` and shows, for each bias:

1. whether the bias was detected and which **modern** method was selected and applied;
2. the full comparison of **every modern and traditional method** on the bias metrics (green = selected modern method,
   blue = best traditional method kept for comparison only);
3. what happened to the 10 records (value changes, removal, or unchanged).

The values come from the actual pipeline run (`preprocessed/sample10_trace.json`); nothing is recomputed here.
After the feature-scale step values are in transformed units.

## Complete details for all 10 sample records

This notebook follows ten fixed-seed training records, not just the tenth row. The detailed tables below show every saved field at every step without a six-value limit. They are a **snapshot of existing saved results**, so no model rerun is required to read them. These explanatory snapshots reflect the saved results at the time of documentation.

Source: [sample10_trace.json](preprocessed/sample10_trace.json). Full flattened details: [sample10_complete_details.csv](preprocessed/sample10_complete_details.csv). Raw and final record tables are included at the end.

**How to read:** `missing (null)` is a recorded missing value; `not captured` means the field was absent from that trace snapshot; `row absent` means the record is no longer present. The previous checkpoint column is explicitly sourced from the previous step's after-snapshot and helps inspect fields omitted when the feature schema changes. It is not a measurement newly produced by this step. Numeric deltas are shown only for fields present on both sides. Learned `z*`/`f*` coordinates cannot be compared numerically with the original clinical features.

The ten records were sampled after splitting, so they do not necessarily illustrate every duplicate, outlier or synthetic row. Global row counts refer to the whole dataset for Duplicate and to the training split thereafter. Synthetic examples inherit metadata, including the source record ID, from a nearest minority record. The trace uses a dictionary keyed only by ID, so a later synthetic row can overwrite an original row snapshot after resampling. Treat those snapshots as ID-level evidence; the final appendix separates original and synthetic exported rows. `target` is the recorded class label; `w` is the model-training weight (exported as `sample_weight`), not a survey-design weight.

| Sample number | Record ID |
|---|---|
| 1 | 1567377 |
| 2 | 1475594 |
| 3 | 1776794 |
| 4 | 840975 |
| 5 | 3001452 |
| 6 | 3155904 |
| 7 | 2869572 |
| 8 | 2369723 |
| 9 | 2731339 |
| 10 | 1665212 |

| Step | Bias | Detected | Applied method | Best traditional | Location |
|---|---|---|---|---|---|
| 1 | Duplicate | No | Skipped | — | Section 3 |
| 2 | Missing data | Yes | MIP-VAE | Random Forest | Section 4 |
| 3 | Outlier | Yes | ECOD | LOF | Section 5 |
| 4 | Data noise | Yes | Robust loss (GCE) | Denoising autoencoder | Section 6 |
| 5 | Feature scale | Yes | SNN (SELU) | Quantile transform | Section 7 |
| 6 | Encoding / domain | Yes | Domain-generalisation transformer | CORAL | Section 8 |
| 7 | Class imbalance | Yes | TabDDPM | SMOTE | Section 9 |
| 8 | Demographic | Yes | Fair contrastive learning | Reweighting | Section 10 |


In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 220); pd.set_option("display.precision", 4)
DATASET = "respiratory"
BASE = Path(r"D:/complete") / DATASET
OUT = BASE / "preprocessed"
trace = json.loads((OUT / "sample10_trace.json").read_text(encoding="utf-8"))
ID_COL, IDS = trace["id_column"], trace["records"]
TABLES = {"Duplicate": "duplicate", "Missing data": "missing_data", "Outlier": "outlier", "Data noise": "data_noise",
          "Feature scale": "feature_scale", "Encoding / domain": "encoding_domain", "Class imbalance": "class_imbalance",
          "Demographic": "demographic"}
TRADITIONAL = {"Duplicate": ["Exact duplicate removal", "Fuzzy matching", "GBER (graph-based ER)"],
               "Missing data": ["GMM", "Random Forest", "GAIN", "MICE", "KNN"],
               "Outlier": ["Isolation Forest", "LOF", "One-Class SVM"],
               "Data noise": ["Denoising autoencoder", "Wavelet denoising", "Median filter", "Savitzky–Golay"],
               "Feature scale": ["Quantile transform", "Min-Max scaler", "Robust scaler"],
               "Encoding / domain": ["DANN", "CORAL", "Autoencoder (domain-conditional)"],
               "Class imbalance": ["SMOTE", "ADASYN", "Borderline-SMOTE", "NearMiss"],
               "Demographic": ["Optimal Transport", "FairSMOTE", "Reweighting"]}


def family(bias, method):
    if "reference" in method.lower() or method.lower().startswith(("no ", "noisy")):
        return "reference (no method)"
    return "Traditional (comparison)" if method in TRADITIONAL[bias] or method.startswith("Savitzky") else "Modern"


def comparison(step):
    path = OUT / f"comparison_{TABLES[step['bias']]}.csv"
    if not path.exists():
        return None
    t = pd.read_csv(path, index_col=0)
    t.insert(0, "family", [family(step["bias"], m) for m in t.index])
    return t


def highlight(t, best, trad):
    return t.style.format(precision=4, na_rep="-").apply(
        lambda r: ["background-color:#e6f6e7;font-weight:bold" if r.name == best else
                   ("background-color:#eef3fc" if r.name == trad else "") for _ in r], axis=1)


def ten_rows(step):
    rows = []
    for rid in IDS:
        b, a = step["before"].get(rid), step["after"].get(rid)
        if b is None and a is None:
            continue
        cols = list(dict.fromkeys(list((b or {}).keys()) + list((a or {}).keys())))
        changed = [c for c in cols if (b or {}).get(c) != (a or {}).get(c)]
        rows.append({"record": rid, "status": "removed by this step" if a is None else ("changed" if changed else "unchanged"),
                     "changed values": ", ".join(f"{c}: {(b or {}).get(c)} -> {(a or {}).get(c)}" for c in changed[:6])
                                       + (f" (+{len(changed) - 6} more)" if len(changed) > 6 else "")})
    return pd.DataFrame(rows).set_index("record")

## The 10 records before preprocessing (raw data)

In [ ]:
raw = pd.read_csv(BASE / "raw" / f"{DATASET}_raw.csv", low_memory=False)
raw[ID_COL] = raw[ID_COL].astype(str)
display(raw[raw[ID_COL].isin(IDS)].set_index(ID_COL).drop(columns=[c for c in raw.columns if c.startswith("meta_") and c != ID_COL]))

## Step 1: Duplicate bias

In [ ]:
step = next(s for s in trace["steps"] if s["bias"] == "Duplicate")
display(Markdown(f"**Detected:** {step['detected']} &nbsp;|&nbsp; **Selected modern method:** {step['selected_modern'] or '- (not detected: nothing applied, before = after)'} "
                 f"&nbsp;|&nbsp; **Best traditional (comparison only):** {step['best_traditional'] or '-'} &nbsp;|&nbsp; training rows {step['rows_before']:,} -> {step['rows_after']:,}"))
t = comparison(step)
if t is not None:
    display(highlight(t, step["selected_modern"], step["best_traditional"]))
display(ten_rows(step))

### Complete step 1 details: Duplicate

Record-level tables remain local; run the notebook with authorized datasets to inspect them.


## Step 2: Missing data bias

In [ ]:
step = next(s for s in trace["steps"] if s["bias"] == "Missing data")
display(Markdown(f"**Detected:** {step['detected']} &nbsp;|&nbsp; **Selected modern method:** {step['selected_modern'] or '- (not detected: nothing applied, before = after)'} "
                 f"&nbsp;|&nbsp; **Best traditional (comparison only):** {step['best_traditional'] or '-'} &nbsp;|&nbsp; training rows {step['rows_before']:,} -> {step['rows_after']:,}"))
t = comparison(step)
if t is not None:
    display(highlight(t, step["selected_modern"], step["best_traditional"]))
display(ten_rows(step))

### Complete step 2 details: Missing data

Record-level tables remain local; run the notebook with authorized datasets to inspect them.


## Step 3: Outlier bias

In [ ]:
step = next(s for s in trace["steps"] if s["bias"] == "Outlier")
display(Markdown(f"**Detected:** {step['detected']} &nbsp;|&nbsp; **Selected modern method:** {step['selected_modern'] or '- (not detected: nothing applied, before = after)'} "
                 f"&nbsp;|&nbsp; **Best traditional (comparison only):** {step['best_traditional'] or '-'} &nbsp;|&nbsp; training rows {step['rows_before']:,} -> {step['rows_after']:,}"))
t = comparison(step)
if t is not None:
    display(highlight(t, step["selected_modern"], step["best_traditional"]))
display(ten_rows(step))

### Complete step 3 details: Outlier

Record-level tables remain local; run the notebook with authorized datasets to inspect them.


## Step 4: Data noise bias

In [ ]:
step = next(s for s in trace["steps"] if s["bias"] == "Data noise")
display(Markdown(f"**Detected:** {step['detected']} &nbsp;|&nbsp; **Selected modern method:** {step['selected_modern'] or '- (not detected: nothing applied, before = after)'} "
                 f"&nbsp;|&nbsp; **Best traditional (comparison only):** {step['best_traditional'] or '-'} &nbsp;|&nbsp; training rows {step['rows_before']:,} -> {step['rows_after']:,}"))
t = comparison(step)
if t is not None:
    display(highlight(t, step["selected_modern"], step["best_traditional"]))
display(ten_rows(step))

### Complete step 4 details: Data noise

Record-level tables remain local; run the notebook with authorized datasets to inspect them.


## Step 5: Feature scale bias

In [ ]:
step = next(s for s in trace["steps"] if s["bias"] == "Feature scale")
display(Markdown(f"**Detected:** {step['detected']} &nbsp;|&nbsp; **Selected modern method:** {step['selected_modern'] or '- (not detected: nothing applied, before = after)'} "
                 f"&nbsp;|&nbsp; **Best traditional (comparison only):** {step['best_traditional'] or '-'} &nbsp;|&nbsp; training rows {step['rows_before']:,} -> {step['rows_after']:,}"))
t = comparison(step)
if t is not None:
    display(highlight(t, step["selected_modern"], step["best_traditional"]))
display(ten_rows(step))

### Complete step 5 details: Feature scale

Record-level tables remain local; run the notebook with authorized datasets to inspect them.


## Step 6: Encoding / domain bias

In [ ]:
step = next(s for s in trace["steps"] if s["bias"] == "Encoding / domain")
display(Markdown(f"**Detected:** {step['detected']} &nbsp;|&nbsp; **Selected modern method:** {step['selected_modern'] or '- (not detected: nothing applied, before = after)'} "
                 f"&nbsp;|&nbsp; **Best traditional (comparison only):** {step['best_traditional'] or '-'} &nbsp;|&nbsp; training rows {step['rows_before']:,} -> {step['rows_after']:,}"))
t = comparison(step)
if t is not None:
    display(highlight(t, step["selected_modern"], step["best_traditional"]))
display(ten_rows(step))

### Complete step 6 details: Encoding / domain

Record-level tables remain local; run the notebook with authorized datasets to inspect them.


## Step 7: Class imbalance bias

In [ ]:
step = next(s for s in trace["steps"] if s["bias"] == "Class imbalance")
display(Markdown(f"**Detected:** {step['detected']} &nbsp;|&nbsp; **Selected modern method:** {step['selected_modern'] or '- (not detected: nothing applied, before = after)'} "
                 f"&nbsp;|&nbsp; **Best traditional (comparison only):** {step['best_traditional'] or '-'} &nbsp;|&nbsp; training rows {step['rows_before']:,} -> {step['rows_after']:,}"))
t = comparison(step)
if t is not None:
    display(highlight(t, step["selected_modern"], step["best_traditional"]))
display(ten_rows(step))

### Complete step 7 details: Class imbalance

Record-level tables remain local; run the notebook with authorized datasets to inspect them.


## Step 8: Demographic bias

In [ ]:
step = next(s for s in trace["steps"] if s["bias"] == "Demographic")
display(Markdown(f"**Detected:** {step['detected']} &nbsp;|&nbsp; **Selected modern method:** {step['selected_modern'] or '- (not detected: nothing applied, before = after)'} "
                 f"&nbsp;|&nbsp; **Best traditional (comparison only):** {step['best_traditional'] or '-'} &nbsp;|&nbsp; training rows {step['rows_before']:,} -> {step['rows_after']:,}"))
t = comparison(step)
if t is not None:
    display(highlight(t, step["selected_modern"], step["best_traditional"]))
display(ten_rows(step))

### Complete step 8 details: Demographic

Record-level tables remain local; run the notebook with authorized datasets to inspect them.


## Why the modern methods were selected

In [ ]:
why = OUT / "modern_vs_traditional.csv"
display(pd.read_csv(why, index_col=0) if why.exists() else "run the main notebook first")

## Raw records, final exported records and record survival

Record-level tables remain local; run the notebook with authorized datasets to inspect them.
